In [1]:
from gensim.models import HdpModel
from gensim.models.callbacks import CoherenceMetric
from gensim import corpora
import tomotopy as tp
from gensim.models.coherencemodel import CoherenceModel
import pandas as pd
import numpy as np
import logging
import sys
import ast
from tqdm import tqdm

In [2]:
def hlda_learning(snum):
    input_data = pd.read_feather(f'../Datasets/sampledf_{snum}.ftr')
    cps = tp.utils.Corpus()

    for i in range(len(input_data['okt'])): 
        if type(input_data['okt'][i]) == float:
            continue
        doc = input_data['okt'][i]
        doc_rm = []
        for word in doc:
            doc_rm.append(word)
        if len(doc_rm) < 1:
            continue
        cps.add_doc(doc_rm)  

    mdl = tp.HLDAModel(tw=tp.TermWeight.ONE, min_df=10, depth=3, corpus=cps)

    mdl.train(0)
    print('Num docs:', len(mdl.docs), ', Vocab size:', len(mdl.used_vocabs), ', Num words:', mdl.num_words)
    print('Removed top words:', mdl.removed_top_words)
    print('Training...', file=sys.stderr, flush=True)
    
    for _ in range(0, 1000, 10):
        mdl.train(7)
        mdl.train(3, freeze_topics=True)
        print('Iteration: {:05}\tll per word: {:.5f}\tNum. of topics: {}'.format(mdl.global_step, mdl.ll_per_word, mdl.live_k))

    for _ in range(0, 100, 10):
        mdl.train(10, freeze_topics=True)
        print('Iteration: {:05}\tll per word: {:.5f}\tNum. of topics: {}'.format(mdl.global_step, mdl.ll_per_word, mdl.live_k))

    mdl.summary(topic_word_top_n=20)
    print('Saving...', file=sys.stderr, flush=True)
    mdl.save(f'../models/hlda_{snum}.bin', True)

In [ ]:
for i in tqdm(range(5)):
    hlda_learning(i)

  0%|                                                                            | 0/10 [00:00<?, ?it/s]Training...
